In [0]:
import requests

# URL apenas com a lista de moedas
url_moedas = "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/Moedas?$top=100&$format=json&$select=simbolo,nomeFormatado"

response = requests.get(url_moedas)
dados_moedas = response.json()['value']

df_moedas = spark.createDataFrame(dados_moedas)

df_moedas = df_moedas.selectExpr("nomeFormatado AS MoedaNome", "simbolo AS Moeda")

df_moedas.write.mode("overwrite").saveAsTable("projeto_bcb.bronze.dim_moedas")

display(df_moedas)
print("Tabela dim_moedas criada com sucesso!")


In [0]:
from pyspark.sql.functions import lit

# 1. Puxando a lista de moedas
df_lista = spark.sql("SELECT Moeda FROM projeto_bcb.bronze.dim_moedas")
lista_moedas = [row['Moeda'] for row in df_lista.collect()]

data_inicial = '01-01-2024'
data_final = '12-31-2025'
top = 100

for moeda in lista_moedas:
    todos_dados = []
    skip = 0
    
    print(f"Extraindo dados da moeda: {moeda}...")
    
    #url
    while True:
        url = (
            f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
            f"CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
            f"@moeda='{moeda}'&@dataInicial='{data_inicial}'&@dataFinalCotacao='{data_final}'"
            f"&$top={top}&$skip={skip}&$filter=tipoBoletim%20eq%20'Fechamento'&$format=json&$select=cotacaoCompra,dataHoraCotacao"
        )
        
        response = requests.get(url)
        
        # Pega os valores, se vier vazio, encerra o while para esta moeda
        dados = response.json().get('value', [])
        if not dados:
            break   
        todos_dados.extend(dados)
        skip += top
        
    # 5. Salvando os dados empilhados desta moeda
    if todos_dados:
        df_cotacoes = spark.createDataFrame(todos_dados).withColumn("moeda", lit(moeda))
        caminho_salvar = f"/Volumes/projeto_bcb/bronze/arquivos_brutos/cotacoes_novos/{moeda}"
        df_cotacoes.write.mode("overwrite").format("parquet").save(caminho_salvar)

print("\n Ingestão completa!")